# 1. Creating Table

## 1.1 Adding Constraints

In [0]:
%sql
CREATE TABLE IF NOT EXISTS cpt_utility_catalog.gold.fct_suburb_water_billing(
    billing_key BIGINT NOT NULL,
    date_key INT NOT NULL,
    suburb_key BIGINT NOT NULL,
    billing_class STRING,
    amount DECIMAL(13,2),
    quantity_kl DECIMAL(13,3),
    number_of_records INT,
    number_of_contract_accounts INT,

    CONSTRAINT pk_fct_suburb_water_billing PRIMARY KEY (billing_key) RELY,
    CONSTRAINT fk_fct_suburb_water_billing_date FOREIGN KEY (date_key) REFERENCES cpt_utility_catalog.gold.dim_date(date_key) RELY,
    CONSTRAINT fk_fct_suburb_water_billing_suburb FOREIGN KEY (suburb_key) REFERENCES cpt_utility_catalog.gold.dim_suburb(suburb_key) RELY

)


## 1.2 Populating Table

In [0]:
%sql
INSERT OVERWRITE TABLE cpt_utility_catalog.gold.fct_suburb_water_billing
SELECT
    xxhash64(b.id) AS billing_key,

    COALESCE(CAST(date_format(b.date, 'yyyyMMdd') AS INT), -1) AS date_key,
    COALESCE(ds.suburb_key, xxhash64('unmapped'))                AS suburb_key,

    b.billing_class,
    b.amount,
    b.quantity_kl,
    b.number_of_records,
    b.number_of_contract_accounts

FROM cpt_utility_catalog.silver.silver_suburb_water_billing_cleaned b

LEFT JOIN cpt_utility_catalog.gold.dim_suburb ds
       ON xxhash64(LOWER(TRIM(b.suburb))) = ds.suburb_key;